# Интерактивная карта мира с уровнем самоубийств (1985-2016)

Этот ноутбук визуализирует данные о самоубийствах по странам с возможностью интерактивного выбора года.

**Источник данных:** [Kaggle - Suicide Rates Overview 1985 to 2016](https://www.kaggle.com/datasets/russellyates88/suicide-rates-overview-1985-to-2016)

## 1. Установка зависимостей

In [9]:
# Установка необходимых библиотек
!pip install pandas plotly numpy

## 2. Импорт библиотек

In [12]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from typing import Tuple
from IPython.display import display

print("✓ Все библиотеки загружены успешно")

✓ Все библиотеки загружены успешно


## 3. Загрузка и подготовка данных

In [2]:
def load_and_prepare_data(csv_path: str) -> pd.DataFrame:
    """
    Загружает данные из CSV файла и подготавливает их для визуализации.
    
    Args:
        csv_path: путь к файлу CSV с данными о самоубийствах
        
    Returns:
        DataFrame с агрегированными данными по странам и годам
    """
    # Загружаем данные
    print("⏳ Загрузка данных...")
    df = pd.read_csv(csv_path)
    
    print(f"✓ Загружено {len(df)} строк данных")
    
    # Агрегируем данные по странам и годам
    # Суммируем количество самоубийств и население для каждой страны в каждый год
    agg_data = df.groupby(['country', 'year']).agg({
        'suicides_no': 'sum',
        'population': 'sum'
    }).reset_index()
    
    # Вычисляем коэффициент самоубийств на 100 тысяч населения
    agg_data['suicide_rate_per_100k'] = (
        agg_data['suicides_no'] / agg_data['population'] * 100000
    )
    
    return agg_data


# Загружаем данные (убедитесь, что файл master.csv находится в текущей директории)
try:
    data = load_and_prepare_data('master.csv')
    print(f"\n✓ Данные подготовлены успешно")
    print(f"  Периоды: {data['year'].min()} - {data['year'].max()}")
    print(f"  Количество стран: {data['country'].nunique()}")
    print(f"  Всего записей: {len(data)}")
except FileNotFoundError:
    print("❌ Ошибка: файл 'master.csv' не найден")
    print("   Пожалуйста, скачайте датасет с Kaggle")

⏳ Загрузка данных...
✓ Загружено 27820 строк данных

✓ Данные подготовлены успешно
  Периоды: 1985 - 2016
  Количество стран: 101
  Всего записей: 2321


## 4. Информация о данных

In [15]:
# Показываем первые строки данных
print("Первые 5 строк агрегированных данных:")
display(data.head())

print("\nОсновная статистика:")
display(data[['suicide_rate_per_100k']].describe())

print("\nТоп 10 стран с наибольшим уровнем самоубийств (за весь период):")
top_countries = data.groupby('country')['suicide_rate_per_100k'].mean().sort_values(ascending=False).head(10)
top_countries

Первые 5 строк агрегированных данных:


,country,year,suicides_no,population,suicide_rate_per_100k
0,Albania,1987,73,2709600,2.694125
1,Albania,1988,63,2764300,2.279058
2,Albania,1989,68,2803100,2.425886
3,Albania,1992,47,2822500,1.665190
4,Albania,1993,73,2807300,2.600363



Основная статистика:


,suicide_rate_per_100k
count,2321.000000
mean,11.738663
std,8.921654
min,0.000000
25%,4.998151
50%,10.083301
75%,16.146591
max,51.019758



Топ 10 стран с наибольшим уровнем самоубийств (за весь период):


country
Lithuania             40.736234
Russian Federation    32.702743
Sri Lanka             30.847564
Belarus               30.231149
Hungary               29.616574
Latvia                27.994640
Kazakhstan            27.040563
Slovenia              26.415561
Estonia               25.744078
Ukraine               24.740542
Name: suicide_rate_per_100k, dtype: float64

## 5. Создание интерактивной карты мира

In [4]:
def create_interactive_map(data: pd.DataFrame, initial_year: int = 2016) -> go.Figure:
    """
    Создает интерактивную карту мира с фреймами для каждого года.
    
    Args:
        data: DataFrame с данными о самоубийствах
        initial_year: начальный год для отображения
        
    Returns:
        Фигура Plotly для интерактивной визуализации
    """
    
    # Получаем список всех лет в возрастающем порядке
    years = sorted(data['year'].unique())
    
    # Создаем фреймы для каждого года
    frames = []
    
    for year in years:
        year_data = data[data['year'] == year]
        
        frame = go.Frame(
            data=[
                go.Choropleth(
                    locations=year_data['country'],
                    z=year_data['suicide_rate_per_100k'],
                    locationmode='country names',
                    colorscale='Reds',
                    text=year_data['country'],
                    hovertemplate='<b>%{text}</b><br>' +
                                  'Уровень самоубийств: %{z:.2f} на 100k<extra></extra>',
                    colorbar=dict(
                        title='Самоубийства<br>на 100k<br>населения',
                        thickness=15,
                        len=0.7,
                        x=1.02
                    ),
                    zmin=data['suicide_rate_per_100k'].min(),
                    zmax=data['suicide_rate_per_100k'].max()
                )
            ],
            name=str(year),
            layout=go.Layout(title_text=f'Уровень самоубийств по странам: {year}')
        )
        frames.append(frame)
    
    # Данные для начального года
    initial_data = data[data['year'] == initial_year]
    
    # Создаем главную фигуру
    fig = go.Figure(
        data=[
            go.Choropleth(
                locations=initial_data['country'],
                z=initial_data['suicide_rate_per_100k'],
                locationmode='country names',
                colorscale='Reds',
                text=initial_data['country'],
                hovertemplate='<b>%{text}</b><br>' +
                              'Уровень самоубийств: %{z:.2f} на 100k<extra></extra>',
                colorbar=dict(
                    title='Самоубийства<br>на 100k<br>населения',
                    thickness=15,
                    len=0.7,
                    x=1.02
                ),
                zmin=data['suicide_rate_per_100k'].min(),
                zmax=data['suicide_rate_per_100k'].max()
            )
        ],
        frames=frames
    )
    
    # Создаем слайдер
    sliders = [
        {
            'active': years.index(initial_year),
            'yanchor': 'top',
            'y': 0,
            'xanchor': 'left',
            'x': 0.1,
            'len': 0.8,
            'transition': {'duration': 300},
            'pad': {'b': 10, 't': 50},
            'currentvalue': {
                'prefix': 'Год: ',
                'visible': True,
                'xanchor': 'center',
                'font': {'size': 16, 'color': '#555'}
            },
            'steps': [
                {
                    'args': [
                        [str(year)],
                        {
                            'frame': {'duration': 300, 'redraw': True},
                            'mode': 'immediate',
                            'transition': {'duration': 300}
                        }
                    ],
                    'label': str(year),
                    'method': 'animate'
                }
                for year in years
            ]
        }
    ]
    
    # Кнопки управления (Play/Pause)
    updatemenus = [
        {
            'type': 'buttons',
            'showactive': False,
            'y': 0,
            'x': 0,
            'xanchor': 'left',
            'yanchor': 'top',
            'pad': {'t': 70, 'r': 10},
            'buttons': [
                {
                    'label': '▶ Проиграть',
                    'method': 'animate',
                    'args': [None, {
                        'frame': {'duration': 500, 'redraw': True},
                        'fromcurrent': True,
                        'transition': {'duration': 300, 'easing': 'quadratic-in-out'}
                    }]
                },
                {
                    'label': '⏸ Пауза',
                    'method': 'animate',
                    'args': [[None], {
                        'frame': {'duration': 0, 'redraw': True},
                        'mode': 'immediate',
                        'transition': {'duration': 0}
                    }]
                }
            ]
        }
    ]
    
    # Обновляем макет
    fig.update_layout(
        title={
            'text': f'Уровень самоубийств по странам: {initial_year}',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'color': '#333'}
        },
        geo=dict(
            projection_type='natural earth',
            bgcolor='rgba(240, 240, 240, 0.5)',
            coastlinecolor='#555',
            showocean=True,
            oceancolor='#e5f3ff'
        ),
        height=700,
        margin=dict(l=0, r=150, t=100, b=100),
        font=dict(family='Arial, sans-serif', size=12),
        sliders=sliders,
        updatemenus=updatemenus,
        hovermode='closest'
    )
    
    return fig


# Создаем интерактивную карту
print("⏳ Создание интерактивной карты...")
fig = create_interactive_map(data, initial_year=2016)
print("✓ Карта создана успешно")

⏳ Создание интерактивной карты...
✓ Карта создана успешно


/tmp/ipykernel_28387/3725502230.py:51: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = go.Figure(


## 6. Визуализация карты

In [5]:
# Показываем интерактивную карту
fig.show()

## 7. Сохранение карты в HTML файл

In [15]:
# Сохраняем карту в HTML файл
output_file = 'suicide_rates_map.html'
fig.write_html(output_file)
print(f"✓ Интерактивная карта сохранена в файл: {output_file}")

✓ Интерактивная карта сохранена в файл: suicide_rates_map.html


## 8. Анализ данных по годам

In [16]:
# Анализируем тренд уровня самоубийств по годам
yearly_stats = data.groupby('year').agg({
    'suicide_rate_per_100k': ['mean', 'min', 'max', 'std']
}).round(2)

print("Статистика по годам:")
yearly_stats

Статистика по годам:


suicide_rate_per_100k                    
                      mean   min    max    std
year                                          
1985                  9.59  0.00  40.90   8.60
1986                  9.51  0.00  37.90   8.11
1987                  9.85  0.00  29.54   7.62
1988                 10.81  0.00  30.26   7.12
1989                 10.98  0.00  30.36   7.44
1990                 10.98  0.00  32.34   7.44
1991                 11.60  0.00  40.92   8.34
1992                 11.85  0.00  41.00   8.42
1993                 12.21  0.59  40.56   8.79
1994                 12.14  0.00  44.35   9.08
1995                 13.81  0.00  50.01  10.98
1996                 13.81  0.38  51.02  11.09
1997                 13.48  0.00  48.51  10.96
1998                 13.52  0.00  46.34  10.56
1999                 13.38  0.00  46.50  10.29
2000                 12.69  0.00  49.20  10.37
2001                 12.47  0.00  46.54  10.05
2002                 12.65  0.00  47.37   9.81
2003                 12.26  0.00  44.71   9.67
2004                 11.63  0.00  42.82   8.85
2005                 11.44  0.00  41.55   8.54
2006                 11.08  0.00  33.56   8.40
2007                 11.22  0.00  33.19   8.27
2008                 11.41  0.00  36.38   8.46
2009                 11.11  0.00  37.73   8.48
2010                 10.86  0.00  34.51   8.18
2011                 10.73  0.00  35.35   7.91
2012                 11.11  0.00  32.67   7.62
2013                 10.98  0.00  38.66   7.84
2014                 10.71  0.00  33.43   6.95
2015                 10.85  0.00  32.54   6.74
2016                 12.90  0.00  33.62   8.41

## 9. График динамики во времени

In [7]:
import plotly.express as px

# Создаем график среднего уровня самоубийств по годам
yearly_avg = data.groupby('year')['suicide_rate_per_100k'].mean().reset_index()

fig_trend = px.line(
    yearly_avg,
    x='year',
    y='suicide_rate_per_100k',
    title='Средний уровень самоубийств в мире (1985-2016)',
    labels={
        'year': 'Год',
        'suicide_rate_per_100k': 'Самоубийства на 100k населения'
    },
    markers=True
)

fig_trend.update_layout(
    height=500,
    hovermode='x unified',
    template='plotly_white'
)

fig_trend.show()

## 10. Страны с наибольшим увеличением/снижением

In [17]:
# Находим изменение за период для каждой страны
country_change = []

for country in data['country'].unique():
    country_data = data[data['country'] == country].sort_values('year')
    if len(country_data) > 0:
        first_year = country_data.iloc[0]['suicide_rate_per_100k']
        last_year = country_data.iloc[-1]['suicide_rate_per_100k']
        change = last_year - first_year
        country_change.append({
            'country': country,
            'change': change,
            'first_year': first_year,
            'last_year': last_year
        })

change_df = pd.DataFrame(country_change).sort_values('change', ascending=False)

print("Топ 10 стран с наибольшим УВЕЛИЧЕНИЕМ уровня самоубийств:")
display(change_df.head(10)[['country', 'change', 'first_year', 'last_year']])

print("\nТоп 10 стран с наибольшим СНИЖЕНИЕМ уровня самоубийств:")
change_df.tail(10)[['country', 'change', 'first_year', 'last_year']]

Топ 10 стран с наибольшим УВЕЛИЧЕНИЕМ уровня самоубийств:


,country,change,first_year,last_year
39,Guyana,22.156723,8.551002,30.707725
60,Montenegro,18.067183,0.000000,18.067183
73,Republic of Korea,17.710174,10.047199,27.757374
99,Uruguay,9.259894,10.484401,19.744296
14,Bosnia and Herzegovina,8.429442,0.136117,8.565559
92,Trinidad and Tobago,8.358190,2.876984,11.235174
56,Malta,7.628777,0.651042,8.279819
13,Belize,6.085595,2.043597,8.129192
24,Cyprus,4.944120,0.116722,5.060843
19,Chile,4.697311,6.397887,11.095198



Топ 10 стран с наибольшим СНИЖЕНИЕМ уровня самоубийств:


,country,change,first_year,last_year
33,France,-9.595346,24.329248,14.733902
26,Denmark,-10.015416,20.492728,10.477312
6,Austria,-13.362588,29.408807,16.046220
32,Finland,-15.432813,29.539889,14.107075
48,Kiribati,-16.166842,16.166842,0.000000
52,Lithuania,-16.397000,50.012562,33.615562
87,Sri Lanka,-18.462763,40.903514,22.440751
40,Hungary,-19.932904,40.921252,20.988347
51,Latvia,-22.486981,43.121236,20.634255
30,Estonia,-28.102115,43.784289,15.682175
